# OCR one book with Mathpix

Run the cells top to bottom. Only **cell 2** needs editing.

**What you need**

- The PDF downloaded to this computer.
- A Mathpix `app_id` and `app_key`.

**What you get**, under `corpus/`:

```
corpus/text/<sha256>/document.mmd     the whole book, Markdown + LaTeX
corpus/text/<sha256>/page-001.md      one file per page
corpus/meta/<sha256>/page-001.json    confidence and length per page
corpus/meta/<sha256>/lines.json       the raw Mathpix response
```

**Rules**

- Do not edit the `.md` files by hand. They get regenerated.
- Mathpix deletes its copy after 30 days. What lands in `corpus/` is the permanent one.
- Run a small page range first (`PAGES = "1-30"`) before doing a whole book.

## 1. Install (first time only)

In [1]:
%pip install --quiet requests pypdf

Note: you may need to restart the kernel to use updated packages.


## 2. Settings — edit this cell

`PAGES = None` does the whole book. `PAGES = "1-30"` does a range, and only those
pages are uploaded and billed.

In [ ]:
# Keys are read from .env, which is gitignored. A notebook gets shared and
# committed; a key pasted into a cell goes with it, and into every export.
import os, re
from pathlib import Path

ENV_FILE = Path(r"c:/Users/96181/bac2/.env")

def from_env(name):
    if os.environ.get(name):
        return os.environ[name]
    if ENV_FILE.exists():
        m = re.search(rf'^\s*{name}\s*=\s*"?([^"\s]+)"?',
                      ENV_FILE.read_text(encoding="utf-8"), re.M)
        if m:
            return m.group(1)
    raise SystemExit(f"{name} is not set. Add it to {ENV_FILE}")

APP_ID  = from_env("MATHPIX_APP_ID")
APP_KEY = from_env("MATHPIX_APP_KEY")

PDF_PATH = r"C:/Users/96181/Downloads/math_gs_1_en.pdf"

PAGES = None          # None = whole book, or "1-30" for a test range

# Leave as None to name the output folder after the PDF file. Set it to
# something readable when the file name is not (e.g. "falsafa-3amma-lh").
BOOK_NAME = None

# Pages per upload. A whole scanned book is a large file, and one long upload
# is the thing most likely to be cut off by the network. Smaller chunks also
# mean a failure costs one chunk, not the book. Drop to 20 on a bad connection.
CHUNK_PAGES = 40

# Absolute, so output lands in the project root. A relative path is resolved
# against the notebook's own folder, which nests it under scripts/corpus/.
OUT_DIR = r"c:/Users/96181/bac2/corpus"

## 3. Setup

In [272]:
import hashlib, io, json, os, re, time, statistics, zipfile
from pathlib import Path

import requests
from pypdf import PdfReader, PdfWriter

API = "https://api.mathpix.com/v3/pdf"
HEADERS = {"app_id": APP_ID, "app_key": APP_KEY}

pdf_path = Path(PDF_PATH)
assert pdf_path.exists(), f"Cannot find {pdf_path}"

raw = pdf_path.read_bytes()
sha256 = hashlib.sha256(raw).hexdigest()
total_pages = len(PdfReader(io.BytesIO(raw)).pages)

def slugify(name):
    "Readable, safe for a folder name on any OS."
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", name).strip("-").lower()
    return slug or "book"

# Folder name = readable slug + the first 8 characters of the hash.
# The slug is so a person can find the book; the hash is so two different files
# that happen to share a name can never overwrite each other, and so every
# folder still traces back to exactly one PDF.
book_slug = slugify(BOOK_NAME or pdf_path.stem)
book_key  = f"{book_slug}__{sha256[:8]}"

text_dir  = Path(OUT_DIR) / "text" / book_key
meta_dir  = Path(OUT_DIR) / "meta" / book_key
fig_dir   = Path(OUT_DIR) / "figures" / book_key
chunk_dir = meta_dir / "chunks"
for d in (text_dir, meta_dir, chunk_dir):
    d.mkdir(parents=True, exist_ok=True)

def page_indices(spec, total):
    "'1-30' or '7' -> zero-based indices. None -> all pages."
    if not spec:
        return list(range(total))
    m = re.fullmatch(r"(\d+)(?:-(\d+))?", spec.strip())
    assert m, f'Cannot read PAGES = "{spec}". Use "12" or "1-30".'
    start = int(m.group(1))
    end = int(m.group(2) or m.group(1))
    assert 1 <= start <= end <= total, f"PAGES {spec} is outside this book (1-{total})."
    return list(range(start - 1, end))

indices = page_indices(PAGES, total_pages)

def subset(page_index_list):
    "Builds a PDF containing only these pages, so each upload stays small."
    writer = PdfWriter()
    reader = PdfReader(io.BytesIO(raw))
    for i in page_index_list:
        writer.add_page(reader.pages[i])
    buf = io.BytesIO()
    writer.write(buf)
    return buf.getvalue()

chunks = [indices[i : i + CHUNK_PAGES] for i in range(0, len(indices), CHUNK_PAGES)]

print(f"book    {book_key}")
print(f"file    {pdf_path.name}  ({len(raw)/1024/1024:.1f} MB)")
print(f"sha256  {sha256}")
print(f"pages   {len(indices)} of {total_pages}")
print(f"uploads {len(chunks)} chunk(s) of up to {CHUNK_PAGES} pages")
print(f"cost    about ${len(indices) * 0.005:.2f}")

book    math-se-en__de339ba1
file    math_se_en.pdf  (32.1 MB)
sha256  de339ba1cb3ad62ee0bbac637be0c02c3cd49dc0460223bc420fc6294cdf26aa
pages   281 of 281
uploads 8 chunk(s) of up to 40 pages
cost    about $1.41


## 4. Helpers — upload, wait, download

Every call retries on network failure. A dropped TLS connection raises an
exception rather than returning a status code, so it has to be caught here or a
single bad moment kills the whole run.

In [273]:
OPTIONS = {
    "rm_spaces": True,
    "math_inline_delimiters": ["$", "$"],
    "math_display_delimiters": ["$$", "$$"],
    "enable_tables_fallback": True,
    # Figures and diagrams are cropped by Mathpix and served from their CDN,
    # which expires. The zip output embeds every referenced image inline, so it
    # is the only self-contained copy. It must be requested HERE, at processing
    # time — it cannot be added to a job that has already run.
    "conversion_formats": {"mmd.zip": True},
}

NETWORK_ERRORS = (requests.exceptions.RequestException,)

def with_retry(what, call, tries=4, wait=10):
    """Runs one HTTP call, retrying on network failure.

    Uploads of a scanned book get cut off — a dropped TLS connection raises
    SSLError rather than returning a status code, so it has to be caught here
    or the whole run dies on one bad moment.
    """
    for attempt in range(1, tries + 1):
        try:
            return call()
        except NETWORK_ERRORS as err:
            if attempt == tries:
                raise
            print(f"    {what}: {type(err).__name__} — retry {attempt}/{tries - 1} in {wait}s")
            time.sleep(wait)
            wait *= 2

def submit(chunk_bytes, name):
    def call():
        r = requests.post(
            API,
            headers=HEADERS,
            files={"file": (name, chunk_bytes, "application/pdf")},
            data={"options_json": json.dumps(OPTIONS)},
            timeout=(30, 600),   # 30s to connect, 10 min to send
        )
        if r.status_code != 200:
            raise SystemExit(f"Upload failed ({r.status_code}): {r.text}")
        return r.json()["pdf_id"]
    return with_retry("upload", call)

def wait_for(pdf_id, label=""):
    "Polls status, not percent_done — Mathpix reports 100% while still assembling."
    started = time.time()
    while True:
        info = with_retry("status", lambda: requests.get(f"{API}/{pdf_id}", headers=HEADERS, timeout=60).json())
        status = info.get("status", "")
        if status == "completed":
            return
        if status == "error":
            raise SystemExit(f"Mathpix reported an error on {label}: {info}")
        print(f"\r    {label} {status} … {int(info.get('percent_done', 0))}%   ", end="")
        if time.time() - started > 45 * 60:
            raise SystemExit(f"Gave up after 45 minutes on {label}. pdf_id {pdf_id}")
        time.sleep(5)

def fetch(pdf_id, ext, tries=8, wait=15, binary=False):
    "Downloads one output format. Conversion finishes after OCR, so it retries."
    for attempt in range(1, tries + 1):
        r = with_retry(f".{ext}", lambda: requests.get(f"{API}/{pdf_id}.{ext}", headers=HEADERS, timeout=600))
        if r.status_code == 200:
            return r.content if binary else r.text
        if attempt < tries:
            time.sleep(wait)
    print(f"    .{ext} unavailable (HTTP {r.status_code})")
    return None

print("helpers ready")

helpers ready


## 5. Read the book, chunk by chunk

This is the long cell. It uploads one chunk at a time and saves each result to
`meta/<book>/chunks/`.

**If it fails part way through, just run this cell again.** Chunks already done
are reused from disk — you do not pay to read them twice.

Do not close the kernel while it runs.

In [274]:
pages, images, mmd_parts = [], [], []

for n, chunk in enumerate(chunks, start=1):
    first, last = chunk[0] + 1, chunk[-1] + 1
    label = f"chunk {n}/{len(chunks)} (pages {first}-{last})"
    cached = chunk_dir / f"chunk-{n:03d}.json"

    # Already done in an earlier run? Reuse it. A network failure half way
    # through a book must not mean paying to read the first half again.
    if cached.exists():
        payload = json.loads(cached.read_text(encoding="utf-8"))
        print(f"  {label}: reusing earlier result")
    else:
        body = subset(chunk)
        print(f"  {label}: uploading {len(body)/1024/1024:.1f} MB …")
        pdf_id = submit(body, f"{book_slug}-{first}-{last}.pdf")
        wait_for(pdf_id, label)

        lines_raw = fetch(pdf_id, "lines.mmd.json") or fetch(pdf_id, "lines.json")
        if not lines_raw:
            raise SystemExit(f"No per-line JSON for {label}. Nothing to check pages against.")

        bundle = fetch(pdf_id, "mmd.zip", binary=True)
        chunk_images, chunk_mmd = [], ""
        if bundle:
            with zipfile.ZipFile(io.BytesIO(bundle)) as z:
                for name in z.namelist():
                    low = name.lower()
                    if low.endswith((".png", ".jpg", ".jpeg", ".gif", ".webp")):
                        fig_dir.mkdir(parents=True, exist_ok=True)
                        out_name = f"p{first:03d}-{Path(name).name}"
                        (fig_dir / out_name).write_bytes(z.read(name))
                        chunk_images.append(out_name)
                    elif low.endswith(".mmd"):
                        chunk_mmd = z.read(name).decode("utf-8")
        else:
            print(f"    {label}: no bundle — figures were NOT captured for these pages")

        payload = {"pdfId": pdf_id, "lines": json.loads(lines_raw),
                   "images": chunk_images, "mmd": chunk_mmd, "firstPage": first}
        cached.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
        print(f"\r  {label}: done, {len(chunk_images)} image(s)                    ")

    # Page numbers come back relative to the uploaded chunk, so shift them to
    # the real page numbers in the book. An off-by-one here misfiles every page
    # after it and surfaces months later as a wrong citation.
    offset = payload["firstPage"]
    for i, page in enumerate(payload["lines"].get("pages") or []):
        lines = page.get("lines") or []
        text = "\n".join(str(l.get("text") or l.get("mmd") or "") for l in lines).strip()
        confs = [float(l["confidence"]) for l in lines if isinstance(l.get("confidence"), (int, float))]
        pages.append({"number": offset + i, "text": text, "confidences": confs})

    images.extend(payload["images"])
    if payload["mmd"]:
        mmd_parts.append(payload["mmd"])

print()
print(f"{len(pages)} pages, {len(images)} images from {len(chunks)} chunk(s)")

  chunk 1/8 (pages 1-40): uploading 4.2 MB …
  chunk 1/8 (pages 1-40): done, 12 image(s)                    
  chunk 2/8 (pages 41-80): uploading 4.8 MB …
  chunk 2/8 (pages 41-80): done, 32 image(s)                    
  chunk 3/8 (pages 81-120): uploading 4.4 MB …
  chunk 3/8 (pages 81-120): done, 24 image(s)                    
  chunk 4/8 (pages 121-160): uploading 4.8 MB …
  chunk 4/8 (pages 121-160): done, 12 image(s)                    
  chunk 5/8 (pages 161-200): uploading 4.7 MB …
  chunk 5/8 (pages 161-200): done, 14 image(s)                    
  chunk 6/8 (pages 201-240): uploading 4.5 MB …
  chunk 6/8 (pages 201-240): done, 8 image(s)                    
  chunk 7/8 (pages 241-280): uploading 4.7 MB …
  chunk 7/8 (pages 241-280): done, 12 image(s)                    
  chunk 8/8 (pages 281-281): uploading 0.1 MB …
    chunk 8/8 (pages 281-281) split … 100%       .mmd.zip unavailable (HTTP 500)
    chunk 8/8 (pages 281-281): no bundle — figures were NOT captured for these 

## 6. Assemble

In [275]:
if mmd_parts:
    (text_dir / "document.mmd").write_text("\n\n".join(mmd_parts), encoding="utf-8")
    print(f"document.mmd   {sum(len(m) for m in mmd_parts):,} characters")
else:
    print("document.mmd   MISSING - the per-page files still carry the LaTeX")

print(f"figures        {len(images)} -> {fig_dir if images else '(none extracted)'}")
print(f"chunks kept    {len(list(chunk_dir.glob('chunk-*.json')))} in {chunk_dir}")
print()
print("The chunk files are the resume point. Delete them only when the book is accepted.")

document.mmd   541,384 characters
figures        114 -> c:\Users\96181\bac2\corpus\figures\math-se-en__de339ba1
chunks kept    8 in c:\Users\96181\bac2\corpus\meta\math-se-en__de339ba1\chunks

The chunk files are the resume point. Delete them only when the book is accepted.


## 7. Split into pages

If this cell says it found no pages, the response shape is different from what
it expects. `meta/lines.json` is already saved — send that file to the team and
the parser gets adjusted.

In [276]:
if not pages:
    raise SystemExit("No pages were produced. Check the chunk output above.")

no_conf = 0
page_means = []
for page in pages:
    stem = f"{page['number']:03d}"
    (text_dir / f"page-{stem}.md").write_text(page["text"], encoding="utf-8")

    confs = page["confidences"]
    if not confs:
        no_conf += 1
    mean_conf = sum(confs) / len(confs) if confs else 1.0
    page_means.append(mean_conf)
    (meta_dir / f"page-{stem}.json").write_text(
        json.dumps(
            {
                "source": {"book": book_key, "file": pdf_path.name, "sha256": sha256, "page": page["number"]},
                "reader": {"provider": "mathpix"},
                "charCount": len(page["text"]),
                "meanConfidence": round(mean_conf, 4),
                "minConfidence": round(min(confs), 4) if confs else 1.0,
                "lowConfidenceTokenRatio": round(sum(c < 0.7 for c in confs) / len(confs), 4) if confs else 0.0,
                "hasConfidence": bool(confs),
                "blocks": [],
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

summary = {
    "book": book_key,
    "file": pdf_path.name,
    "sha256": sha256,
    "reader": "mathpix",
    "pdfPageCount": len(indices),
    "pagesProcessed": len(pages),
    "meanConfidence": round(sum(page_means) / len(page_means), 4),
    "images": len(images),
    "chunks": len(chunks),
    "processedAt": time.strftime("%Y-%m-%d %H:%M"),
}
(meta_dir / "document.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

# One index of every book processed, so "which books are done?" is a file to
# read rather than a folder listing to interpret.
index_path = Path(OUT_DIR) / "books.json"
index = json.loads(index_path.read_text(encoding="utf-8")) if index_path.exists() else {}
index[book_key] = summary
index_path.write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")

Path(OUT_DIR, ".last-book").write_text(book_key, encoding="utf-8")

print(f"wrote {len(pages)} page files to {text_dir}")
print(f"books processed so far: {len(index)}")
if no_conf:
    print(f"NOTE: {no_conf} pages came back with no confidence values.")
    print("      Those score 1.0, so the confidence check is not protecting them.")

wrote 281 page files to c:\Users\96181\bac2\corpus\text\math-se-en__de339ba1
books processed so far: 25
NOTE: 16 pages came back with no confidence values.
      Those score 1.0, so the confidence check is not protecting them.


## 8. Check the pages

Nothing here is corrected automatically. The point is to find the pages a human
should look at.

For prose the failure that matters is not a wrong letter — it is a **skipped or
duplicated paragraph**, which is invisible in the text itself. That is what the
length and repetition checks are for.

In [277]:
PRESENTATION_FORMS = re.compile("[ﭐ-﷿ﹰ-﻿]")
TATWEEL            = re.compile("ـ")
ARABIC_DIGITS      = re.compile("[٠-٩۰-۹]")

MEAN_CONFIDENCE_MIN = 0.85
SHORT_FACTOR        = 0.40   # of this book's median page length
LONG_FACTOR         = 2.50
REPEAT_NGRAM        = 8      # words
REPEAT_LIMIT        = 3      # times one 8-word run may appear
REPEAT_COVERAGE     = 0.15   # and it must cover this much of the page

def repetition(text, n=REPEAT_NGRAM):
    words = text.split()
    if len(words) < n * 2:
        return 0, 0.0
    counts = {}
    for i in range(len(words) - n + 1):
        gram = " ".join(words[i : i + n])
        counts[gram] = counts.get(gram, 0) + 1
    worst = max(counts.values())
    return worst, (worst * n) / len(words)

median_chars = statistics.median([len(p["text"]) for p in pages]) or 1
flagged = []

for page in pages:
    text, confs = page["text"], page["confidences"]
    mean_conf = sum(confs) / len(confs) if confs else 1.0
    flags = []

    if mean_conf < MEAN_CONFIDENCE_MIN:
        flags.append("low-confidence")
    if not text.strip():
        flags.append("empty")
    elif len(text) < median_chars * SHORT_FACTOR:
        flags.append("short-page")
    elif len(text) > median_chars * LONG_FACTOR:
        flags.append("long-page")

    count, coverage = repetition(text)
    if count > REPEAT_LIMIT and coverage > REPEAT_COVERAGE:
        flags.append("repetition-loop")

    if PRESENTATION_FORMS.search(text): flags.append("presentation-forms")
    if TATWEEL.search(text):            flags.append("tatweel")
    if ARABIC_DIGITS.search(text):      flags.append("arabic-indic-digits")

    if flags:
        flagged.append((page["number"], len(text), round(mean_conf, 3), flags))

print(f"{book_key}")
print(f"  file           {pdf_path.name}")
print(f"  pages          {len(pages)} of {len(indices)} sent" + ("" if len(pages) == len(indices) else "   <-- MISMATCH, tell the team"))
print(f"  images         {len(images)}")
print(f"  median length  {int(median_chars)} characters")
print(f"  flagged        {len(flagged)} of {len(pages)} ({len(flagged)/len(pages)*100:.1f}%)")
print()

tally = {}
for _, _, _, flags in flagged:
    for f in flags:
        tally[f] = tally.get(f, 0) + 1
for flag, count in sorted(tally.items(), key=lambda kv: -kv[1]):
    print(f"    {flag:<22} {count}")

if flagged:
    print()
    print("  pages to look at:")
    for number, chars, conf, flags in flagged[:40]:
        print(f"    p{number:>4}  {chars:>5} chars  conf {conf:.3f}  {', '.join(flags)}")
    if len(flagged) > 40:
        print(f"    … and {len(flagged) - 40} more")

math-se-en__de339ba1
  file           math_se_en.pdf
  pages          281 of 281 sent
  images         114
  median length  1929 characters
  flagged        36 of 281 (12.8%)

    short-page             16
    empty                  15
    low-confidence         6
    repetition-loop        1

  pages to look at:
    p   1    237 chars  conf 0.819  low-confidence, short-page
    p   2      0 chars  conf 1.000  empty
    p   3    118 chars  conf 1.000  short-page
    p   4     61 chars  conf 1.000  short-page
    p   5    248 chars  conf 1.000  short-page
    p   6    223 chars  conf 0.997  short-page
    p   8      1 chars  conf 0.434  low-confidence, short-page
    p  12      0 chars  conf 1.000  empty
    p  26      0 chars  conf 1.000  empty
    p  48      0 chars  conf 1.000  empty
    p  61    320 chars  conf 1.000  short-page
    p  62      0 chars  conf 1.000  empty
    p  80    231 chars  conf 0.932  short-page
    p  88    718 chars  conf 0.973  short-page
    p 101    758 cha

## 9. Read a page yourself

In [278]:
PAGE_TO_READ = first_page_number      # change this to any page number

print((text_dir / f"page-{PAGE_TO_READ:03d}.md").read_text(encoding="utf-8")[:3000])

\title{
Building up
 MOTHEmartICS
}


Secondary Education

Third Year

Sociology and Economics Section

![](https://cdn.mathpix.com/cropped/44e19e33-2020-4b2f-9da4-17b75becd53f-01.jpg?height=1440&width=1786&top_left_y=1000&top_left_x=95)


## 9b. Build one readable document

Turns the per-page text into two files you can actually read:

- **`document.md`** — every page in order, with page markers. This is the file the
  next stage of the pipeline reads.
- **`document.html`** — the same content with the equations rendered, so a teacher
  can compare it against the PDF side by side. Open it in any browser.

The HTML needs internet the first time it opens (it pulls MathJax from a CDN to
draw the equations).

In [279]:
SECTION  = re.compile(r"\\(?:sub)?section\*?\{([^}]*)\}")
TABULAR  = re.compile(r"\\begin\{tabular\}(?:\[[^\]]*\])?\{[^}]*\}(.*?)\\end\{tabular\}", re.S)

def tabular_to_html(match):
    """LaTeX tabular -> HTML table.

    MathJax does not draw `tabular` (it is a text-mode environment, not maths),
    so a maths book full of tables would render as raw backslashes. Converting
    them here is the difference between a reviewable page and a wall of markup.
    """
    body = re.sub(r"\\hline", "", match.group(1))
    rows = [row.strip() for row in body.split(r"\\") if row.strip()]
    cells = ["<tr>" + "".join(f"<td>{c.strip()}</td>" for c in row.split("&")) + "</tr>" for row in rows]
    return "<table>" + "".join(cells) + "</table>"

def to_paragraph(block):
    """Wraps one block for HTML.

    Line breaks become <br>, except in blocks containing display maths: MathJax
    reads a $$...$$ span from a single text node, and a <br> element in the
    middle of one leaves the equation unrendered on the page. Inside maths the
    row breaks come from \\\\ anyway, so the newlines are not needed.
    """
    if block.startswith(("<table", "<h3")):
        return block
    inner = block if "$$" in block else block.replace("\n", "<br>\n")
    return "<p>" + inner + "</p>"

md_parts, html_parts = [], []
for page in pages:
    number, text = page["number"], page["text"]

    md_parts.append(f"<!-- page {number} -->\n\n{text}\n")

    body = TABULAR.sub(tabular_to_html, text)
    body = SECTION.sub(r"<h3>\1</h3>", body)
    blocks = [b.strip() for b in body.split("\n\n") if b.strip()]
    rendered = "\n".join(to_paragraph(b) for b in blocks)
    html_parts.append(f'<section><div class="page">page {number}</div>\n{rendered}\n</section>')

# document.md — the machine-readable copy, and what the next stage reads.
doc_md = f"<!-- {pdf_path.name} | {sha256} -->\n\n" + "\n\n---\n\n".join(md_parts)
(text_dir / "document.md").write_text(doc_md, encoding="utf-8")

# document.html — the human-readable copy, equations drawn by MathJax.
doc_html = """<!doctype html>
<html><head><meta charset="utf-8">
<title>__TITLE__</title>
<script>window.MathJax={tex:{inlineMath:[["$","$"]],displayMath:[["$$","$$"]]}};</script>
<script async src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>
<style>
 body{font-family:Georgia,serif;max-width:820px;margin:2rem auto;padding:0 1rem;line-height:1.6}
 section{border-top:1px solid #ddd;padding-top:1rem;margin-top:2rem}
 .page{font:12px monospace;color:#999;margin-bottom:.5rem}
 table{border-collapse:collapse;margin:1rem 0}
 td{border:1px solid #bbb;padding:.25rem .6rem;font-size:.9rem}
 h3{font-size:1.1rem;color:#1f3b57}
</style></head><body>
<h1>__TITLE__</h1>
<p style="color:#777;font-size:.85rem">__SUB__</p>
__BODY__
</body></html>"""
doc_html = (doc_html
            .replace("__TITLE__", pdf_path.name)
            .replace("__SUB__", f"pages {pages[0]['number']}&ndash;{pages[-1]['number']} &middot; {sha256[:12]}")
            .replace("__BODY__", "\n".join(html_parts)))
(text_dir / "document.html").write_text(doc_html, encoding="utf-8")

print(f"document.md    {len(doc_md):,} characters")
print(f"document.html  {len(doc_html):,} characters")
print()
print("Open this in a browser to check the equations:")
print(" ", (text_dir / "document.html").resolve())

document.md    564,081 characters
document.html  635,890 characters

Open this in a browser to check the equations:
  C:\Users\96181\bac2\corpus\text\math-se-en__de339ba1\document.html


## 10. Before you call the book done

- [ ] Pages written = pages sent.
- [ ] Opened 5 random pages above and compared them against the PDF.
- [ ] Flagged share is under 10%.
- [ ] Recorded the book, sha256, page count, and flagged share in the tracking sheet.
- [ ] Sent the output of cell 8 to the team.

Do not hand-correct pages. Flag them and move on — someone else handles the flagged list.

`presentation-forms`, `tatweel` and `arabic-indic-digits` are expected on Arabic
books at this stage; a later step normalises them. They are recorded, not problems.